# EvoCA — test notebook

Tests the C core (ctypes), verifies exact GoL, and launches the SDL2 interactive display.

Run cells in order.  **Build first**, then proceed.

## 1  Build

In [ ]:
import subprocess, sys, os
root = os.path.abspath('')
# macOS: gcc -dynamiclib; Linux: gcc -shared
import platform
flag = '-dynamiclib' if platform.system() == 'Darwin' else '-shared'
ext  = 'dylib'      if platform.system() == 'Darwin' else 'so'
cmd  = ['gcc', '-O2', '-Wall', '-fPIC', flag,
        '-o', f'C/libevoca.{ext}', 'C/evoca.c']
r = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(r.stdout or '(no stdout)')
if r.returncode != 0:
    print('STDERR:', r.stderr, file=sys.stderr)
    raise RuntimeError('Build failed')
print('Build OK')

## 2  Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import time

from python.evoca_py import EvoCA, make_gol_lut, LUT_BYTES, lut_bit_index, unpack_lut
from python.display  import run as sdl_run

## 3  GoL LUT — correctness check

The LUT is indexed by **(v_x, n1, n2, n3, n4, n5)** where n1..n5 are per-ring
active-cell counts.  GoL depends only on `(v_x, n1+n2)`; outer-ring counts are
ignored.  This is exact — no approximation.

**Why per-ring counts instead of weighted sum?**  
The original spec proposed `S_x = A + B√2 + C√5` with `A = n1+2n3`.  
This conflates n1 and n3: the same A can arise from e.g. `(n1=1,n3=1)` *or*
`(n1=3,n3=0)`, giving different Moore counts (1 vs 3).  GoL cannot distinguish
these cases, so a single LUT entry would have to be wrong for one of them.  
Switching to separate per-ring counts removes the ambiguity.

LUT size: 2×5×5×5×9×5 = **11 250 bits** = **1 407 bytes/cell** (bit-packed).  
Memory: N=256 → 92 MB; N=512 → 369 MB.

In [ ]:
gol_lut = make_gol_lut()
bits    = unpack_lut(gol_lut)
print(f'LUT size: {LUT_BYTES} bytes')

all_ok = True
for v_x in (0, 1):
    for moore in range(9):
        n1 = min(moore, 4); n2 = moore - n1
        if n2 > 4: continue
        got  = bits[lut_bit_index(v_x, n1, n2, 0, 0, 0)]
        want = (1 if moore == 3 else 0) if v_x == 0 else (1 if moore in (2,3) else 0)
        ok   = got == want
        all_ok &= ok
        print(f'  v={v_x} Moore={moore:2d}  LUT={got}  expected={want}  {"OK" if ok else "FAIL"}')

# outer ring ignored
assert bits[lut_bit_index(1, 2, 0, 3, 5, 2)] == 1
assert bits[lut_bit_index(0, 2, 1, 4, 8, 4)] == 1
print('\nOuter-ring-ignore checks OK')
print('\nAll GoL LUT entries correct:', all_ok)

## 4  Glider test — exact GoL

A standard GoL glider has period 4 and moves (1,1) diagonally per period.
With the correct LUT this must be exact — no approximation.

In [ ]:
N = 64
sim = EvoCA()
sim.init(N, food_inc=0.0, m_scale=0.0, food_repro=1e9)  # food/repro disabled
sim.set_lut_all(gol_lut)
sim.set_cgenom_all(0); sim.set_f_all(0.0); sim.set_F_all(0.0)

grid = np.zeros((N, N), dtype=np.uint8)
r0, c0 = 10, 10
for dr, dc in [(0,1),(1,2),(2,0),(2,1),(2,2)]:
    grid[r0+dr, c0+dc] = 1
sim.set_v(grid)

STEPS = 40
frames = [sim.get_v().copy()]
for _ in range(STEPS):
    sim.step()
    frames.append(sim.get_v().copy())

alive = [f.sum() for f in frames]
print('Alive count per step (should stay at 5):', alive[:12])
assert all(a == 5 for a in alive), f'Glider lost cells: {alive}'

# verify diagonal travel
pos0 = np.argwhere(frames[0]).mean(0)
pos4 = np.argwhere(frames[4]).mean(0)
shift = pos4 - pos0
print(f'4-step shift: {shift}  (expect [1, 1])')
assert abs(shift[0]-1) < 0.1 and abs(shift[1]-1) < 0.1
print('Glider test PASSED — exact GoL confirmed')

In [ ]:
# Static snapshots
show_at = [0, 1, 2, 3, 4, 8, 12, 20, 40]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, t in zip(axes.ravel(), show_at):
    ax.imshow(frames[t], cmap='binary', interpolation='nearest', vmin=0, vmax=1)
    ax.set_title(f't = {t}')
    ax.axis('off')
plt.suptitle('EvoCA — exact GoL glider (B3/S23)')
plt.tight_layout()
plt.show()

In [ ]:
# Animation
fig2, ax2 = plt.subplots(figsize=(5,5))
im    = ax2.imshow(frames[0], cmap='binary', interpolation='nearest', vmin=0, vmax=1)
title = ax2.set_title('t = 0'); ax2.axis('off')
def _upd(t):
    im.set_data(frames[t]); title.set_text(f't = {t}'); return im, title
ani = animation.FuncAnimation(fig2, _upd, frames=len(frames), interval=120, blit=True)
plt.close(fig2)
HTML(ani.to_jshtml())

## 5  Performance benchmark

In [ ]:
sim.free()
rng = np.random.default_rng(42)
for N_test in (128, 256, 512):
    s = EvoCA(); s.init(N_test, 0, 0, 1e9)
    s.set_lut_all(gol_lut)
    s.set_v(rng.integers(0, 2, (N_test, N_test), dtype=np.uint8))
    s.set_cgenom_all(0); s.set_f_all(0); s.set_F_all(0)
    NSTEPS = 20
    t0 = time.perf_counter()
    for _ in range(NSTEPS): s.step()
    dt = time.perf_counter() - t0
    print(f'N={N_test:4d}:  {NSTEPS/dt:6.1f} fps  ({dt/NSTEPS*1000:.1f} ms/step)')
    s.free()

## 6  SDL2 interactive display

Launches a separate SDL2 window with:
- Grid display (color mode: cell state / env food / private food)
- Metaparam widget strip (food_inc, m_scale, food_repro)

**Keyboard**: `SPACE` pause/unpause · `C` cycle color · `S` single step · `Q`/`Esc` quit  
**Mouse**: click widget row to adjust parameters; click color/pause/quit blocks

This cell **blocks** until the window is closed.

In [ ]:
N = 256
rng2 = np.random.default_rng(0)

sim_sdl = EvoCA()
sim_sdl.init(N, food_inc=0.01, m_scale=0.5, food_repro=0.8)
sim_sdl.set_lut_all(gol_lut)
sim_sdl.set_cgenom_all(0b000000)
sim_sdl.set_v(rng2.integers(0, 2, (N, N), dtype=np.uint8))
sim_sdl.set_f_all(0.1)
sim_sdl.set_F_all(0.5)

sdl_run(sim_sdl)   # blocks until window closed
sim_sdl.free()

## 7  GoL-only SDL2 display  (food/repro disabled)

Use this to observe pure GoL dynamics with mutation=0.

In [ ]:
N = 512
rng3 = np.random.default_rng(7)

sim_gol = EvoCA()
sim_gol.init(N, food_inc=0.0, m_scale=0.0, food_repro=1e9)
sim_gol.set_lut_all(gol_lut)
sim_gol.set_cgenom_all(0)
sim_gol.set_v(rng3.integers(0, 2, (N, N), dtype=np.uint8))
sim_gol.set_f_all(0.0); sim_gol.set_F_all(0.0)

sdl_run(sim_gol)   # blocks until window closed
sim_gol.free()